# mmpR5 Pipeline — Controller
Orchestrates **m1 → m5** via [papermill](https://papermill.readthedocs.io/).

| Step | What happens |
|------|-------------|
| 1 | Edit the config cell below |
| 2 | Run **m0_setup_and_discovery.ipynb** manually in a separate tab (ColabFold install + restart) |
| 3 | Run All Cells in this notebook |

> **m0 is NOT executed here** — ColabFold installation requires a runtime restart
> which terminates any running notebook.  After completing m0 manually, a
> `colabfold_ready.flag` sentinel is written to Drive; this controller checks for it
> before running m3.

## 0 · Install papermill

In [ ]:
import subprocess, sys
_pkgs = ["papermill", "ipykernel"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _pkgs, check=True)
print("papermill ready")

## 1 · Pipeline Configuration
> **✏️ Edit this cell, then Run All.**

In [ ]:
# ── Edit here ──────────────────────────────────────────────────────────────────
# Input samples
SAMPLE_CSV    = ""          # path on Drive to CSV with srr,sample_label,phenotype
SINGLE_SRR    = ""          # single SRR accession (used when SAMPLE_CSV is empty)
SAMPLE_LABEL  = ""          # label for single SRR

# Drive paths
DRIVE_OUTPUT  = "mmpR5_pipeline/output"
NB_DIR        = "mmpR5_pipeline/notebooks"   # where the m*.ipynb files live on Drive

# Pipeline settings
RUN_ML        = True         # set False to skip m4 if <20 labeled samples
REPORT_TITLE  = "mmpR5 Pipeline Run"
MAX_PER_FIGURE = 12

# Module skip flags  (True = skip)
SKIP_M1 = False
SKIP_M2 = False
SKIP_M3 = False
SKIP_M4 = False
SKIP_M5 = False
# ── End edit ───────────────────────────────────────────────────────────────────

## 2 · Mount Drive & Write Config

In [ ]:
from google.colab import drive
import json, os
from pathlib import Path

drive.mount("/content/drive", force_remount=False)
DRIVE_BASE = Path("/content/drive/MyDrive")

_cfg = {
    "SAMPLE_CSV":     SAMPLE_CSV,
    "SINGLE_SRR":     SINGLE_SRR,
    "SAMPLE_LABEL":   SAMPLE_LABEL,
    "DRIVE_OUTPUT":   DRIVE_OUTPUT,
    "NB_DIR":         NB_DIR,
    "RUN_ML":         RUN_ML,
    "REPORT_TITLE":   REPORT_TITLE,
    "MAX_PER_FIGURE": MAX_PER_FIGURE,
}
_cfg_path = DRIVE_BASE / "mmpR5_pipeline" / "pipeline_config.json"
_cfg_path.parent.mkdir(parents=True, exist_ok=True)
_cfg_path.write_text(json.dumps(_cfg, indent=2))
print(f"Config written → {_cfg_path}")
print(json.dumps(_cfg, indent=2))

## 3 · Check ColabFold Sentinel

In [ ]:
_sentinel = DRIVE_BASE / "mmpR5_pipeline" / "colabfold_ready.flag"
if not _sentinel.exists():
    raise RuntimeError(
        "\n" + "="*60 + "\n"
        "ColabFold is NOT installed.\n\n"
        "Please:\n"
        "  1. Open m0_setup_and_discovery.ipynb in a new tab\n"
        "  2. Run it fully (it will install ColabFold & restart the runtime)\n"
        "  3. After the runtime restarts, re-run m0 from the post-install cell\n"
        "  4. Verify the sentinel is written, then return here and Run All\n"
        + "="*60
    )
print(f"✓ Sentinel found: {_sentinel}")
print(f"  Written: {__import__('datetime').datetime.fromtimestamp(_sentinel.stat().st_mtime)}")

## 4 · Run Modules via papermill

In [ ]:
import papermill as pm
import traceback, datetime

_NB_DIR  = DRIVE_BASE / NB_DIR
_OUT_DIR = DRIVE_BASE / DRIVE_OUTPUT / "executed_notebooks"
_OUT_DIR.mkdir(parents=True, exist_ok=True)

_status = {}  # module_name -> "ok" | "skipped" | "error: ..."

def _run_module(name, params, skip=False):
    """Execute a module notebook via papermill; record status."""
    if skip:
        print(f"[SKIP] {name}")
        _status[name] = "skipped"
        return
    _nb_in  = _NB_DIR / f"{name}.ipynb"
    _nb_out = _OUT_DIR / f"{name}_executed_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.ipynb"
    if not _nb_in.exists():
        msg = f"Notebook not found: {_nb_in}"
        print(f"[ERROR] {msg}")
        _status[name] = f"error: {msg}"
        return
    print(f"\n{'─'*60}")
    print(f"▶  {name}  ({datetime.datetime.now().strftime('%H:%M:%S')})")
    print(f"   input  : {_nb_in}")
    print(f"   output : {_nb_out}")
    print(f"   params : {params}")
    try:
        pm.execute_notebook(
            str(_nb_in),
            str(_nb_out),
            parameters=params,
            kernel_name="python3",
            progress_bar=True,
        )
        _status[name] = "ok"
        print(f"✓  {name} completed")
    except pm.exceptions.PapermillExecutionError as _e:
        _status[name] = f"error: {_e.ename} — {str(_e.evalue)[:200]}"
        print(f"[ERROR] {name} failed:\n{_status[name]}")
        traceback.print_exc()
    except Exception as _e:
        _status[name] = f"error: {type(_e).__name__} — {str(_e)[:200]}"
        print(f"[ERROR] {name} failed:\n{_status[name]}")
        traceback.print_exc()

_base_params = {
    "DRIVE_OUTPUT": DRIVE_OUTPUT,
    "SAMPLE_CSV":   SAMPLE_CSV,
    "SINGLE_SRR":   SINGLE_SRR,
    "SAMPLE_LABEL": SAMPLE_LABEL,
}

print("Starting pipeline execution...")
print(f"Notebook dir : {_NB_DIR}")
print(f"Output dir   : {_OUT_DIR}")

### m1 — Download & QC

In [ ]:
_run_module("m1_data_acquisition", {**_base_params}, skip=SKIP_M1)

### m2 — Assembly & Variants

In [ ]:
_run_module("m2_assembly_and_extraction", {**_base_params}, skip=SKIP_M2)

### m3 — Structure & API Annotations

In [ ]:
_run_module("m3_structural_features", {**_base_params}, skip=SKIP_M3)

### m4 — ML Classification

In [ ]:
_run_module("m4_ml_classification", {**_base_params, "RUN_ML": RUN_ML}, skip=SKIP_M4)

### m5 — Reporting

In [ ]:
_run_module("m5_reporting", {
    **_base_params,
    "REPORT_TITLE":   REPORT_TITLE,
    "MAX_PER_FIGURE": MAX_PER_FIGURE,
}, skip=SKIP_M5)

## 5 · Run Summary

In [ ]:
import datetime

print("\n" + "="*60)
print(f"  Pipeline complete — {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)
_icons = {"ok": "✓", "skipped": "○"}
for _mod, _st in _status.items():
    _ic = _icons.get(_st, "✗")
    print(f"  {_ic}  {_mod:35s}  {_st}")
print()

_n_ok  = sum(1 for v in _status.values() if v == "ok")
_n_sk  = sum(1 for v in _status.values() if v == "skipped")
_n_err = sum(1 for v in _status.values() if v.startswith("error"))
print(f"  Completed: {_n_ok}   Skipped: {_n_sk}   Errors: {_n_err}")

if _n_err:
    print("\n[!] One or more modules failed. Check the executed notebooks above for details.")
else:
    print("\nAll modules finished successfully.")
    print(f"Results in Drive → {DRIVE_OUTPUT}")